# Pràctica progressiva — Anàlisi de vendes d'una botiga

## Encàrrec professional
Treballes com a tècnic/a d'anàlisi de dades d'una botiga. La persona responsable comercial t'ha lliurat un CSV de vendes i necessita un resum fiable per prendre decisions. Hauràs de preparar les dades, justificar les regles aplicades i comunicar els resultats a un públic no tècnic.

## Objectius
- Llegir CSV amb `csv.DictReader` i explorar-ne l'estructura.
- Validar, normalitzar i convertir dades amb funcions pròpies.
- Separar dades vàlides, corregibles i rebutjades amb traçabilitat.
- Calcular indicadors i fer agrupacions.
- Verificar els resultats i generar fitxers d'informe.
- Formular conclusions prudents i defensar les decisions.

**Fitxers:** posa `vendes_botiga.csv` al mateix directori que aquest notebook.

La pràctica es divideix en fases guiades. Utilitza només la biblioteca estàndard de Python.

## Fase 1 — Inspecció guiada

Comença per entendre el fitxer abans de modificar-lo. Cada fila és una línia de venda; els camps són textuals quan es llegeixen del CSV.

In [1]:
import csv
from datetime import datetime

nom_fitxer = "vendes_botiga.csv"

# EXERCI 1.0: Obre el fitxer 'nom_fitxer' en mode lectura, llegeix-lo amb csv.DictReader
# i desa els camps a la variable 'camps' i els registres com una llista a 'registres'.

with open(nom_fitxer, mode="r", encoding="utf-8", newline="") as fitxer:
    lector = csv.DictReader(fitxer)
    camps = lector.fieldnames
    registres = list(lector)

# Comprovacions
print("Camps del CSV:", camps)
print("Nombre total de registres:", len(registres))
if registres:
    print("Exemple de primer registre:", registres[0])

Camps del CSV: ['data', 'producte', 'categoria', 'unitats', 'preu_unitari', 'ciutat']
Nombre total de registres: 19
Exemple de primer registre: {'data': '2026-01-03', 'producte': 'Llibreta', 'categoria': 'Papereria', 'unitats': '12', 'preu_unitari': '2.50', 'ciutat': 'Barcelona'}


### Exercici 1.1
Calcula quants registres tenen almenys un camp buit o format només per espais. No modifiquis les dades encara.

**Pista:** Pots fer servir `.strip() == ""` sobre cada valor del diccionari per comprovar si un camp està buit o té només espais.

In [2]:
import csv

# GUIA PAS A PAS:
# 1. Inicialitza el comptador 'registres_incomplets' a 0.
# 2. Revisa cada registre de la llista 'registres'.
# 3. Comprova si algun dels seus valors està buit després d'aplicar .strip().
# 4. Incrementa el comptador si es detecta algun camp buit.

registres = []
with open("vendes_botiga.csv", mode="r", encoding="utf-8", newline="") as fitxer:
    lector = csv.DictReader(fitxer)
    for fila in lector:
        registres.append(fila)

registres_incomplets = 0

for reg in registres:
    # TODO: Comprova si hi ha algun camp buit en la fila reg
    # Pista: utilitza any(...) o un bucle sobre reg.values()
    if any(str(val).strip() == "" for val in reg.values()):
        registres_incomplets += 1

print(f"Registres amb almenys un camp buit/espais: {registres_incomplets}")

Registres amb almenys un camp buit/espais: 2


### Exercici 1.2

**Reflexiona:** quines dades són imprescindibles per calcular l'import d'una venda? Quines són necessàries per segmentar-la/classificar-la?

## Fase 2 — Regles de qualitat i neteja guiada

Cal aplicar als registres de vendes aquestes regles:
1. Eliminar espais sobrants dels camps textuals (`strip()`).
2. `unitats` ha de ser un enter estrictament positiu (> 0).
3. `preu_unitari` ha de ser un nombre positiu (> 0); s'accepten coma o punt decimal.
4. La data ha de tenir el format `AAAA-MM-DD` i ser una data vàlida.
5. Producte, categoria i ciutat són camps descriptius obligatoris.
6. Un registre amb valors numèrics o data no interpretables no entra als càlculs. No inventis valors per completar-los (imputació).

### Exercici 2 — Funcions de conversió amb gestió d'errors
Implementa funcions de conversió robustes utilitzant blocs `try...except`.

In [3]:
def converteix_enter_positiu(valor: str):
    """Retorna un int positiu (>0) o None si no és vàlid."""
    try:
        v = int(str(valor).strip())
        return v if v > 0 else None
    except (ValueError, TypeError):
         return None

def converteix_preu(valor: str):
    """Accepta coma o punt decimal; retorna float positiu (>0) o None si no és vàlid."""
    # TODO: Neteja espais, reemplaça ',' per '.', converteix a float i comprova que sigui > 0
    try:
        preu = str(valor).strip().replace(',', '.')
        v = float(preu)
        return v if v > 0 else None
    except (ValueError, TypeError):
        return None

def data_valida(valor: str) -> bool:
    """Retorna True si el text correspon a una data vàlida en format AAAA-MM-DD."""
    # TODO: Utilitza datetime.strptime(..., "%Y-%m-%d") dins d'un try-except
    try:
        datetime.strptime(str(valor).strip(), "%Y-%m-%d")
        return True
    except (ValueError, TypeError):
        return False

### Exercici 3 — Neteja amb traçabilitat

Classifica les dades en tres estructures:
- `vendes_netes`: llista de diccionaris amb dades convertides i camp `import` inclòs.
- `rebutjades`: llista de registres no vàlids.
- `incidencies`: llista de registres/incidències amb el detall per a la traçabilitat (`fila_csv`, `tipus`, `camp`, `valor_original`, `accio`).

In [4]:
from datetime import datetime

vendes_netes = []
incidencies = []
rebutjades = []

obligatoris = ["producte", "categoria", "ciutat"]

for idx, reg in enumerate(registres, start=2): # Comencem a 2 per la capçalera del CSV
    es_valid = True
    reg_net = {}
    
    # TODO PAS 1: Comprovació de camps obligatoris
    # - Comprova si producte, categoria i ciutat estan buits.
    # - Si estan buits: marca es_valid = False i afegeix una incidència 'REBUTJAT'.
    # - Si tenen espais de més: afegeix una incidència 'CORRECCIO' i guarda el text net a reg_net.
    for camp in obligatoris:
            val_original = reg.get(camp, "")
            val_net = str(val_original).strip()
            if not val_net:
                es_valid = False
                incidencies.append({
                    "fila_csv": idx, "tipus": "REBUTJAT", "camp": camp,
                    "valor_original": val_original, "accio": "Camp obligatori buit"
                })
            else:
                if val_original != val_net:
                    incidencies.append({
                        "fila_csv": idx, "tipus": "CORRECCIO", "camp": camp,
                        "valor_original": val_original, "accio": "Espais sobrants eliminats"
                    })
                reg_net[camp] = val_net
    
    # TODO PAS 2: Conversió d'unitats
    # - Utilitza converteix_enter_positiu(reg.get("unitats"))
    # - Si és None: es_valid = False i afegeix incidència 'REBUTJAT'.
    # - Si és vàlid: desa el valor convertir a reg_net["unitats"].
    val_unitats = reg.get("unitats")
    unitats_conv = converteix_enter_positiu(val_unitats)
    if unitats_conv is None:
        es_valid = False
        incidencies.append({
            "fila_csv": idx, "tipus": "REBUTJAT", "camp": "unitats",
            "valor_original": val_unitats, "accio": "Unitats no vàlides o no positives"
        })
    else:
        reg_net["unitats"] = unitats_conv

    # TODO PAS 3: Conversió de preu unitari
    # - Utilitza converteix_preu(reg.get("preu_unitari"))
    # - Detecta si tenia coma per afegir una incidència 'CORRECCIO'.
    # - Si és None: es_valid = False i afegeix incidència 'REBUTJAT'.
    val_preu = reg.get("preu_unitari")
    preu_conv = converteix_preu(val_preu)
    if val_preu is not None and "," in str(val_preu):
        incidencies.append({
            "fila_csv": idx, "tipus": "CORRECCIO", "camp": "preu_unitari",
            "valor_original": val_preu, "accio": "Coma substituïda per punt decimal"
        })
        
    if preu_conv is None:
        es_valid = False
        incidencies.append({
            "fila_csv": idx, "tipus": "REBUTJAT", "camp": "preu_unitari",
            "valor_original": val_preu, "accio": "Preu unitari no vàlid o no positiu"
        })
    else:
        reg_net["preu_unitari"] = preu_conv
    
    # TODO PAS 4: Validació de data
    # - Utilitza data_valida(reg.get("data"))
    # - Si no és vàlida: es_valid = False i afegeix incidència 'REBUTJAT'.
    val_data = reg.get("data")
    if not data_valida(val_data):
        es_valid = False
        incidencies.append({
            "fila_csv": idx, "tipus": "REBUTJAT", "camp": "data",
            "valor_original": val_data, "accio": "Data no vàlida o format incorrecte"
        })
    else:
        reg_net["data"] = str(val_data).strip()

    # TODO PAS 5: Classificació final
    # - Si es_valid és True: calcula reg_net["import"] = round(unitats * preu, 2)
    #   i afegeix reg_net a vendes_netes.
    # - Si es_valid és False: afegeix reg a rebutjades.
    if es_valid:
        reg_net["import"] = round(reg_net["unitats"] * reg_net["preu_unitari"], 2)
        vendes_netes.append(reg_net)
    else:
        rebutjades.append(reg)

print(f"Llegits: {len(registres)}")
print(f"Vàlids: {len(vendes_netes)}")
print(f"Rebutjats: {len(rebutjades)}")
print(f"Incidències documentades: {len(incidencies)}")

Llegits: 19
Vàlids: 16
Rebutjats: 3
Incidències documentades: 5


## Fase 3 — Indicadors de negoci

Calcula els indicadors globals i realitza les agrupacions per categoria i ciutat.

In [43]:
# GUIA DE CÀLCUL:
# 1. Suma total d'unitats i d'imports a partir de 'vendes_netes'.
# 2. Càlcul de mitjanes: preu unitari mitjà vs import mitjà per línia.
# 3. Agrupació en diccionaris acumulant la facturació per clau.

# TODO: Calcula els indicadors totals
total_unitats = sum(v["unitats"] for vendes in vendes_netes)
facturacio_total = sum(v["import"] for vendes in vendes_netes)
preu_mitja = facturacio_total / total_unitats if total_unitats > 0 else 0
import_mitja = facturacio_total / len(vendes_netes) if len(vendes_netes) > 0 else 0

# TODO: Agrupa la facturació per categoria i ciutat utilitzant diccionaris
resum_categoria = {}
resum_ciutat = {}

for v in vendes_netes:
    cat = v["categoria"]
    ciu = v["ciutat"]
    resum_categoria[cat] = round(resum_categoria.get(cat, 0.0) + v["import"], 2)
    resum_ciutat[ciu] = round(resum_ciutat.get(ciu, 0.0) + v["import"], 2)

print(f"Total unitats: {total_unitats}")
print(f"Facturació total: {facturacio_total} €")
print(f"Preu unitari mitjà: {preu_mitja} €")
print(f"Import mitjà per línia: {import_mitja} €")
print("Facturació per categoria:", resum_categoria)
print("Facturació per ciutat:", resum_ciutat)

Total unitats: 96
Facturació total: 1776.0 €
Preu unitari mitjà: 18.5 €
Import mitjà per línia: 111.0 €
Facturació per categoria: {'Papereria': 150.5, 'Accessoris': 317.4, 'Electrònica': 498.7}
Facturació per ciutat: {'Barcelona': 551.3, 'Girona': 189.0, 'Tarragona': 201.3, 'Lleida': 25.0}


### Control de qualitat
Validació automàtica de la consistència de les dades mitjançant assercions (`assert`). Omple la informació corresponent per verificar les teves dades.

In [5]:
# TODO 1: Comprova que la suma de vàlids i rebutjats sigui igual al total llegit
assert len(vendes_netes) + len(rebutjades) == len(registres), "Error: La suma de vàlids i rebutjats no quadrat amb el total."

# TODO 2: Comprova que tots els valors de unitats, preu_unitari i import siguin positius (>0)
# assert all(...), "Error: Hi ha valors nuls o negatius"
assert all(v["unitats"] > 0 for v in vendes_netes), "Error: Hi ha unitats nuls o negatius"
assert all(v["preu_unitari"] > 0 for v in vendes_netes), "Error: Hi ha preus unitaris nuls o negatius"
assert all(v["import"] > 0 for v in vendes_netes), "Error: Hi ha imports nuls o negatius"
# TODO 3: Comprova que la suma del diccionari resum_categoria quadri amb facturacio_total
# assert abs(sum(resum_categoria.values()) - facturacio_total) < 0.01, "Error en la suma de categories"
# Calcula els indicadors totals
total_unitats = sum(venda["unitats"] for venda in vendes_netes)
facturacio_total = sum(venda["import"] for venda in vendes_netes)
preu_mitja = facturacio_total / total_unitats if total_unitats > 0 else 0
import_mitja = facturacio_total / len(vendes_netes) if vendes_netes else 0

print("Executa aquesta cel·la quan hagis completat les assercions!")

Executa aquesta cel·la quan hagis completat les assercions!


## Fase 4 — Interpretació i comunicació

Respon a les preguntes analítiques identificant els màxims i avaluant les conclusions de negoci.

In [46]:
# TODO 1: Troba la categoria i ciutat amb més facturació
# Pista: pots fer servir max(resum_categoria.items(), key=lambda x: x[1])
top_categoria = max(resum_categoria.items(), key=lambda x: x[1]) if resum_categoria else (None, 0)
top_ciutat = max(resum_ciutat.items(), key=lambda x: x[1]) if resum_ciutat else (None, 0)

# TODO 2: Troba el producte amb més unitats venudes
# Pista: necessitaràs agrupar primer les unitats totals venudes per producte
unitats_per_producte = {}
# recorre vendes_netes per anar sumant les unitats venudes de cada producte, ves guardant els totals per producte al diccionari unitats_per_producte
for v in vendes_netes:
    prod = v["producte"]
    unitats_per_producte[prod] = unitats_per_producte.get(prod, 0) + v["unitats"]
top_producte = max(unitats_per_producte.items(), key=lambda x: x[1]) if unitats_per_producte else (None, 0)

print(f"1. Categoria amb més facturació: {top_categoria[0]} ({top_categoria[1]} €)")
print(f"   Ciutat amb més facturació: {top_ciutat[0]} ({top_ciutat[1]} €)")
print(f"2. Producte amb més unitats venudes: {top_producte[0]} ({top_producte[1]} unitats)")

1. Categoria amb més facturació: Electrònica (498.7 €)
   Ciutat amb més facturació: Barcelona (551.3 €)
2. Producte amb més unitats venudes: Bolígraf (65 unitats)


## Fase 5 — Lliurament professional

Generació dels tres fitxers de sortida: `informe_vendes.txt`, `incidencies.csv` i `vendes_netes.csv`.

In [8]:
nom_informe = "informe_vendes.txt"
nom_incidencies = "incidencies.csv"
nom_netes = "vendes_netes.csv"

# TODO 1: Genera 'vendes_netes.csv' utilitzant csv.DictWriter amb les dades de vendes_netes
if vendes_netes:
    with open(nom_netes, mode="w", encoding="utf-8", newline="") as f:
        escriptor = csv.DictWriter(f, fieldnames=vendes_netes[0].keys())
        escriptor.writeheader()
        escriptor.writerows(vendes_netes)

# TODO 2: Genera 'incidencies.csv' utilitzant csv.DictWriter amb els camps:
# ["fila_csv", "tipus", "camp", "valor_original", "accio"] 
if incidencies:
    with open(nom_incidencies, mode="w", encoding="utf-8", newline="") as f:
        camps_incidencies = ["fila_csv", "tipus", "camp", "valor_original", "accio"]
        escriptor = csv.DictWriter(f, fieldnames=camps_incidencies)
        escriptor.writeheader()
        escriptor.writerows(incidencies)

# TODO 3: Genera l'informe de text 'informe_vendes.txt' amb un resum executiu que inclogui:
# - Data de generació
# - Cobertura de dades (vàlids vs rebutjats)
# - Indicadors globals (unitats, facturació, mitjanes)
# - Desglossament per categoria i ciutat 
with open(nom_informe, mode="w", encoding="utf-8") as f:
    f.write("==================\n")
    f.write("INFORME DE VENDES\n")
    f.write("==================\n")
    f.write(f"Data de generació: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"1. COBERTURA DE DADES:\n")
    f.write(f"Total llegits: {len(registres)}\n")
    f.write(f"Total vàlids: {len(vendes_netes)}\n")
    f.write(f"Total rebutjats: {len(rebutjades)}\n")
    f.write(f"Incidències documentades: {len(incidencies)}\n")

    f.write(f"2. INDICADORS GLOBALS:\n")
    f.write(f"Total unitats venudes: {total_unitats}\n")
    f.write(f"Facturació total: {facturacio_total:.2f} €\n")
    f.write(f"Preu unitari mitjà: {preu_mitja:.2f} €\n")
    f.write(f"Import mitjà per línea: {import_mitja:.2f} €\n")

    f.write(f"3. DESGLOSSAMENT PER CATEGORIA:\n")
    
    for venda in vendes_netes:
        categoria = venda["categoria"]
        ciutat = venda["ciutat"]
        import_venda = venda["import"]

        resum_categoria[categoria] = resum_categoria.get(categoria, 0) + import_venda
        resum_ciutat[ciutat] = resum_ciutat.get(ciutat, 0) + import_venda

    for cat, fact in resum_categoria.items():
        f.write(f"Categoria: {cat}, Facturació: {fact:.2f} €\n")
    f.write("\n")

    f.write(f"4. DESGLOSSAMENT PER CIUTAT:\n")
    for ciu, fact in resum_ciutat.items():
        f.write(f"Ciutat: {ciu}, Facturació: {fact:.2f} €\n")

print("Fitxers d'informe i exportació generats correctament.")

Fitxers d'informe i exportació generats correctament.
